# FinReason — AWQ quantize + vLLM serve (Colab Pro)

Runtime: **A100 40GB + High-RAM** (Runtime -> Change runtime type).

Pipeline: merge LoRA into base -> AWQ 4-bit -> push to Hub -> serve with vLLM -> test.
Run cells top to bottom. ~20-30 min. Stop the runtime when done.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Install deps
If autoawq/vllm versions clash, this is where it shows.

In [ ]:
!pip install -q autoawq vllm transformers peft accelerate openai huggingface_hub

## 2. HF login (need a WRITE token: huggingface.co/settings/tokens)

In [ ]:
from huggingface_hub import login
login()  # paste write token

## 3. Merge adapter into base -> one full fp16 model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE    = "Qwen/Qwen2.5-7B-Instruct"
ADAPTER = "glen-louis/finreason-qwen2.5-7b-dpo"
MERGED  = "finreason-merged"

base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER)
model = model.merge_and_unload()
model.save_pretrained(MERGED, safe_serialization=True)
AutoTokenizer.from_pretrained(BASE).save_pretrained(MERGED)
del base, model; torch.cuda.empty_cache()
print('merged ->', MERGED)
!du -sh finreason-merged   # ~14GB fp16 (record this)

## 4. AWQ quantize (16-bit -> 4-bit)

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

AWQ_OUT = "finreason-qwen2.5-7b-awq"
quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}

awq_model = AutoAWQForCausalLM.from_pretrained("finreason-merged", device_map="auto")
tok = AutoTokenizer.from_pretrained("finreason-merged")
awq_model.quantize(tok, quant_config=quant_config)
awq_model.save_quantized(AWQ_OUT)
tok.save_pretrained(AWQ_OUT)
print('awq ->', AWQ_OUT)
!du -sh finreason-qwen2.5-7b-awq   # ~5GB (record this -> memory reduction number)

## 5. Push to Hub

In [ ]:
HF_REPO = "glen-louis/finreason-qwen2.5-7b-awq"
awq_model.model.push_to_hub(HF_REPO)
tok.push_to_hub(HF_REPO)
print('pushed ->', HF_REPO)

## 6. Serve with vLLM (background) + test
vLLM blocks, so launch it in the background, wait for load, then hit it.

In [ ]:
import subprocess, time
srv = subprocess.Popen([
    "vllm", "serve", "finreason-qwen2.5-7b-awq",
    "--quantization", "awq", "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.90", "--port", "8000"
])
print('starting vLLM... wait ~60s for model load')
time.sleep(75)

In [ ]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="x")
SYSTEM = "You are a financial analyst. Reason step-by-step and end with a line 'Final Answer:'."
Q = "Revenue grew from $4.2B in 2019 to $5.7B in 2020. What percent growth?"
r = client.chat.completions.create(
    model="finreason-qwen2.5-7b-awq",
    messages=[{"role":"system","content":SYSTEM},{"role":"user","content":Q}],
    temperature=0.0, max_tokens=512)
print(r.choices[0].message.content)

## 7. (optional) vLLM Prometheus metrics + throughput

In [ ]:
!curl -s http://localhost:8000/metrics | grep -E 'vllm:(num_requests|prompt_tokens|generation_tokens)' | head

## Capture for the portfolio, then stop the runtime
- merged size ~14GB vs AWQ ~5GB  -> memory reduction
- the answer above ending 'Final Answer:'  -> screenshot
- vLLM /metrics  -> throughput / token counts

**Runtime -> Disconnect and delete runtime** to free the GPU.